# Attention：从透明 oracle 到框架对账

**适合读者**：已经理解矩阵乘法、准备把 Attention 公式和框架实现对上的学习者。

**先修**：softmax、张量 shape；需要 NumPy、PyTorch 与 JAX，不需要 GPU。

**路线**：先预测 causal mask → 查看 NumPy oracle → PyTorch/JAX 对账 → 移除 mask/缩放制造反例。

**完成信号**：能解释右上三角为什么必须为零，以及三框架数值一致为什么不等于 kernel 性能一致。

## 1. 先预测，再看 NumPy oracle

在运行下一格前画一个 4×4 矩阵：第 0 行应看见几个 key？第 3 行呢？再预测右上三角的概率应该是 0、很小，还是不确定。

本格固定随机输入并返回完整 probability matrix，因此适合做数学 oracle；它不代表 GPU kernel 的内存或速度。

In [ ]:
import numpy as np

from about_llm.from_scratch.attention_numpy import causal_mask, scaled_dot_product_attention

rng = np.random.default_rng(42)
query = rng.normal(size=(1, 4, 8)).astype(np.float32)
key = rng.normal(size=(1, 4, 8)).astype(np.float32)
value = rng.normal(size=(1, 4, 6)).astype(np.float32)
mask = causal_mask(4)
numpy_output, numpy_probabilities = scaled_dot_product_attention(query, key, value, mask=mask)
print('mask:\n', mask.astype(int))
print('probabilities:\n', np.round(numpy_probabilities[0], 3))
assert np.all(numpy_probabilities[0][np.triu_indices(4, k=1)] == 0)


## 2. 用 PyTorch 重写同一公式

逐项对照 score shape、缩放、mask 方向和 softmax 轴。若结果不同，先定位第一处张量差异，不要立刻归因于“框架精度”。

In [ ]:
import torch

q_t, k_t, v_t = map(torch.from_numpy, (query, key, value))
scores_t = q_t @ k_t.transpose(-2, -1) / (q_t.shape[-1] ** 0.5)
scores_t = scores_t.masked_fill(~torch.from_numpy(mask), -torch.inf)
torch_output = torch.softmax(scores_t, dim=-1) @ v_t
np.testing.assert_allclose(torch_output.numpy(), numpy_output, rtol=1e-5, atol=1e-6)
print('PyTorch matches NumPy:', torch_output.shape)


## 3. 再换 JAX，但不改变问题

JAX 使用同一组 Q/K/V 和同一个 NumPy mask。三方一致只证明这个固定输入下的前向公式对齐，不证明 backend、dtype 或 fused kernel 等价。

In [ ]:
import jax
import jax.numpy as jnp

scores_j = jnp.asarray(query) @ jnp.swapaxes(jnp.asarray(key), -2, -1) / (query.shape[-1] ** 0.5)
scores_j = jnp.where(jnp.asarray(mask), scores_j, -jnp.inf)
jax_output = jax.nn.softmax(scores_j, axis=-1) @ jnp.asarray(value)
np.testing.assert_allclose(np.asarray(jax_output), numpy_output, rtol=1e-5, atol=1e-6)
print('JAX matches NumPy:', jax_output.shape)


## 4. 故意破坏两个关键条件

先移除 causal mask，观察未来概率质量；再保留 mask 但移除 1/√d 缩放，观察概率分布变化。先写下你的预测，再运行。

In [ ]:
_, unmasked_probabilities = scaled_dot_product_attention(query, key, value)
future_mass = float(unmasked_probabilities[0][np.triu_indices(4, k=1)].sum())

unscaled_scores = query @ np.swapaxes(key, -2, -1)
unscaled_scores = np.where(mask, unscaled_scores, -np.inf)
unscaled_scores -= np.max(unscaled_scores, axis=-1, keepdims=True)
unscaled_probabilities = np.exp(unscaled_scores)
unscaled_probabilities /= unscaled_probabilities.sum(axis=-1, keepdims=True)
scale_delta = float(np.max(np.abs(unscaled_probabilities - numpy_probabilities)))

print('future probability mass without causal mask:', round(future_mass, 6))
print('max probability change without 1/sqrt(d):', round(scale_delta, 6))
# 不要断言 future_mass > 0：softmax 输出恒为正，这个条件即使 mask 完全正确也成立，
# 抓不到任何 bug。真正的对照是「有 mask 时严格为 0，没 mask 时明显不为 0」。
masked_future_mass = float(numpy_probabilities[0][np.triu_indices(4, k=1)].sum())
assert masked_future_mass == 0.0
assert future_mass > 0.1
assert scale_delta > 1e-3

## 5. 解释结果，而不只记录“通过”

- mask 反例说明 causal 不变量必须靠测试锁定；shape 正确并不足够。
- 缩放反例说明 1/√d 会改变 softmax 尺度，但单个随机样例不能证明训练稳定性。
- 三框架对账是数值正确性证据，不是 GPU 性能证据。

**练习**：改变 head dimension，先预测缩放前后 score 方差和 softmax entropy，再运行；随后加入 fully masked row。

**答案脚手架**：分别记录 `score.var()`、`-(p * log(p)).sum()` 与 `isfinite`。head dimension 增大时，不缩放 score 的方差通常变大；fully masked row 若没有显式策略，softmax 很可能产生 NaN，因此不能把它当作普通有效行。